In [ ]:
import numpy as np
import torch
from torch import nn

In [39]:
class Attention(nn.Module):
    def __init__(self, d_model, attn_dim):
        super().__init__()

        self.d_model = d_model
        self.attn_dim = attn_dim 

        self.scaling_factor = attn_dim ** 0.5

        self.W_q = nn.Linear(d_model, attn_dim, bias=False)
        self.W_k = nn.Linear(d_model, attn_dim, bias=False)
        self.W_v = nn.Linear(d_model, attn_dim, bias=False)

    def forward(self, x, mask=None):
        queries = self.W_q(x)
        keys = self.W_k(x)

        scores = queries @ keys.transpose(-2, -1)
        scores = scores / self.scaling_factor

        if mask is not None:
            keys_mask = mask.unsqueeze(1)  # (B, 1, T)
            scores = scores.masked_fill(keys_mask == 0, float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)

        values = self.W_v(x)
        attn_output = attn_weights @ values
        return attn_output

class MultiheadAttention(nn.Module):
    def __init__(self, d_model, attn_dim, num_heads):
        super().__init__()

        self.d_model = d_model
        self.attn_dim = attn_dim
        self.num_heads = num_heads

        self.heads = nn.ModuleList([Attention(d_model, attn_dim) for _ in range(num_heads)])
        self.output_projection = nn.Linear(attn_dim * num_heads, d_model)

    def forward(self, x, mask=None):
        # print("Input shape:", x.shape)  # Debug print
        head_outputs = [head(x, mask) for head in self.heads]
        # print("Head outputs shapes:", [h.shape for h in head_outputs])  # Debug print
        concatenated = torch.cat(head_outputs, dim=-1)
        # print("Concatenated shape:", concatenated.shape)  # Debug print
        output = self.output_projection(concatenated)
        # print("Output shape:", output.shape)  # Debug print
        return output

class TransformerBlock(nn.Module):
    def __init__(self, d_model, attn_dim, num_heads, ff_dim=1024):
        super().__init__()
        self.attn = MultiheadAttention(d_model, attn_dim, num_heads)
        self.norm1 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.attn(x, mask))
        x = self.norm2(x + self.ffn(x))
        return x
    

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)  # (max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)  # (max_len, 1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) *
            (-torch.log(torch.tensor(10000.0)) / d_model)
        )  # (d_model/2,)

        pe[:, 0::2] = torch.sin(position * div_term)  # even dims
        pe[:, 1::2] = torch.cos(position * div_term)  # odd dims

        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]

class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, max_seq_len=256, d_model=512):
        super().__init__()
        pad_id = vocab_size
        self.embedding = nn.Embedding(vocab_size + 1, d_model, padding_idx=pad_id)
        self.pos_encoding = SinusoidalPositionalEncoding(d_model, max_seq_len)

        self.layer_1 = TransformerBlock(d_model, 64, 8)
        self.layer_2 = TransformerBlock(d_model, 64, 8)

        self.cls_head = nn.Sequential(
            nn.Linear(d_model, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1)
        )

    def forward(self, x, mask=None):
        # x: (batch, seq)
        batch_size, seq_len = x.shape
        
        embd = self.embedding(x)
        # print("Embedding shape:", embd.shape)  # Debug print
        embd = self.pos_encoding(embd)


        out = self.layer_1(embd, mask)

        out = self.layer_2(out, mask)
        # print("After layer 2 shape:", out.shape)  # Debug print

        if mask is not None:
            mask = mask.unsqueeze(-1).float()      # (B, T, 1)
            out = out * mask
            pooled = out.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        else:
            pooled = out.mean(dim=1)
        # print("Pooled shape:", pooled.shape)  # Debug print
        return self.cls_head(pooled)

## Breaking down 

In [ ]:
true_strings: list[str] = []
corrupted_strings: list[str] = []
with open("challenge-data/train.txt", "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t", maxsplit=1)
        if len(parts) < 2:
            continue
        true_strings.append(parts[0])
        corrupted_strings.append(parts[1])

In [ ]:
from bpe_tokenizer import BPETokenizer

tokenizer = BPETokenizer()
tt = tokenizer.load("bpe_tokenizer.json")

In [ ]:
tt.encode(true_strings[0])

In [40]:
model = TransformerClassifier(vocab_size=tt.vocab.keys().__len__())
# model(torch.tensor([tt.encode(true_strings[0])]))

f

In [12]:
max_len = float("-inf")
min_len = float("inf")
tokenized = []
for s in true_strings:
    enc = tt.encode(s)
    tokenized.append(enc)
    if len(enc) > max_len:
        max_len = len(enc)

    if len(enc) < min_len:
        min_len = len(enc)
print("Max sequence length:", max_len)
print("Min sequence length:", min_len)

Max sequence length: 1398
Min sequence length: 2


In [14]:
buckets_bounds = np.arange(0, max_len + 50, 50)
buckets_bounds

array([   0,   50,  100,  150,  200,  250,  300,  350,  400,  450,  500,
        550,  600,  650,  700,  750,  800,  850,  900,  950, 1000, 1050,
       1100, 1150, 1200, 1250, 1300, 1350, 1400])

In [15]:
buckets = {b: [] for b in buckets_bounds}
for t in tokenized:
    l = len(t)
    for b in buckets_bounds:
        if l <= b:
            buckets[b].append(t)
            break

for k, v in buckets.items():
    print(f"Bucket {k}: {len(v)} sequences")

Bucket 0: 0 sequences
Bucket 50: 829484 sequences
Bucket 100: 160790 sequences
Bucket 150: 8527 sequences
Bucket 200: 852 sequences
Bucket 250: 194 sequences
Bucket 300: 71 sequences
Bucket 350: 34 sequences
Bucket 400: 17 sequences
Bucket 450: 9 sequences
Bucket 500: 8 sequences
Bucket 550: 5 sequences
Bucket 600: 3 sequences
Bucket 650: 0 sequences
Bucket 700: 0 sequences
Bucket 750: 1 sequences
Bucket 800: 0 sequences
Bucket 850: 1 sequences
Bucket 900: 3 sequences
Bucket 950: 0 sequences
Bucket 1000: 0 sequences
Bucket 1050: 0 sequences
Bucket 1100: 0 sequences
Bucket 1150: 0 sequences
Bucket 1200: 0 sequences
Bucket 1250: 0 sequences
Bucket 1300: 0 sequences
Bucket 1350: 0 sequences
Bucket 1400: 1 sequences


In [18]:
tt._id2tok[0]

'\x00'

In [27]:
from torch.nn.utils.rnn import pad_sequence

B = 32
n = 50
fifty = buckets[n][:B]

# 1. Convert each sequence in the list to a tensor individually
tensor_list = [torch.tensor(seq, dtype=torch.long) for seq in fifty]

# 2. Use a dummy tensor of length 50 to force the padding width
dummy = torch.zeros(n, dtype=torch.long)
batch = pad_sequence(tensor_list + [dummy], batch_first=True, padding_value=len(tt.vocab))

# 3. Slice off the dummy and create the mask
batch = batch[:B] 
batch_masks = (batch != len(tt.vocab)).long()  # Mask: 1 for real tokens, 0 for padding

print("Batch shape:", batch.shape)      # torch.Size([32, 50])
print("Batch mask shape:", batch_masks.shape) # torch.Size([32, 50])

print("Batch:", batch[1])
print("Batch masks:", batch_masks[1])


Batch shape: torch.Size([32, 50])
Batch mask shape: torch.Size([32, 50])
Batch: tensor([  68,  525,  275, 3675,  296,  318,  303, 2650,  275, 3524,  370, 1412,
         425,  296, 1799,  318, 1394,  808,  281,  370, 1497,  281,  260, 3201,
        2482,  301, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096,
        4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096,
        4096, 4096])
Batch masks: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0])


In [28]:
import random
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.nn.utils.rnn import pad_sequence


class ClassificationDataset(Dataset):
    """
    Stores all (token_ids, label) pairs flat.
    Also exposes per-bucket index lists (split by class) for the sampler.

    label 0 = true string
    label 1 = corrupted string
    """

    def __init__(self, true_tokenized: list[list[int]], corrupted_tokenized: list[list[int]],
                 bucket_size: int = 50):
        # flat list of (ids, label)
        self.samples: list[tuple[list[int], int]] = (
            [(ids, 0) for ids in true_tokenized] +
            [(ids, 1) for ids in corrupted_tokenized]
        )

        # bucket_size-wide length buckets → {bucket_ceil: {"true": [...idx], "corrupt": [...idx]}}
        max_len = max(len(ids) for ids, _ in self.samples)
        bounds = list(range(bucket_size, max_len + bucket_size, bucket_size))
        self.buckets: dict[int, dict[str, list[int]]] = {
            b: {"true": [], "corrupt": []} for b in bounds
        }
        for i, (ids, label) in enumerate(self.samples):
            for b in bounds:
                if len(ids) <= b:
                    key = "true" if label == 0 else "corrupt"
                    self.buckets[b][key].append(i)
                    break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids, label = self.samples[idx]
        return torch.tensor(ids, dtype=torch.long), label


class BucketBatchSampler(Sampler):
    """
    Each batch: B/2 true + B/2 corrupted, all from the same length bucket.
    Iterates over all populated buckets, shuffling within each.
    """

    def __init__(self, dataset: ClassificationDataset, batch_size: int, shuffle: bool = True):
        assert batch_size % 2 == 0, "batch_size must be even"
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.half = batch_size // 2

    def _make_batches(self):
        batches = []
        for b, split in self.dataset.buckets.items():
            true_idx = split["true"][:]
            corrupt_idx = split["corrupt"][:]
            if self.shuffle:
                random.shuffle(true_idx)
                random.shuffle(corrupt_idx)
            # pair up half-batches from each class
            n = min(len(true_idx) // self.half, len(corrupt_idx) // self.half)
            for i in range(n):
                batch = (
                    true_idx[i * self.half : (i + 1) * self.half] +
                    corrupt_idx[i * self.half : (i + 1) * self.half]
                )
                if self.shuffle:
                    random.shuffle(batch)
                batches.append(batch)
        if self.shuffle:
            random.shuffle(batches)
        return batches

    def __iter__(self):
        yield from self._make_batches()

    def __len__(self):
        total = 0
        for split in self.dataset.buckets.values():
            n = min(len(split["true"]) // self.half, len(split["corrupt"]) // self.half)
            total += n
        return total


def collate_fn(batch, pad_id: int):
    """Pad to the longest sequence in the batch and build a boolean mask."""
    seqs, labels = zip(*batch)
    padded = pad_sequence(seqs, batch_first=True, padding_value=pad_id)   # (B, T)
    mask = (padded != pad_id).long()                                       # (B, T)
    labels = torch.tensor(labels, dtype=torch.long)
    return padded, mask, labels


# ── Build dataset ──────────────────────────────────────────────────────────────
print("Tokenizing true strings...")
true_tokenized = [tt.encode(s) for s in true_strings]
print("Tokenizing corrupted strings...")
corrupted_tokenized = [tt.encode(s) for s in corrupted_strings]

dataset = ClassificationDataset(true_tokenized, corrupted_tokenized, bucket_size=50)
print(f"Dataset size: {len(dataset):,} samples")

# ── Build DataLoader ───────────────────────────────────────────────────────────
BATCH_SIZE = 32
PAD_ID = len(tt.vocab)   # one past the last real token id

sampler = BucketBatchSampler(dataset, batch_size=BATCH_SIZE, shuffle=True)
loader = DataLoader(
    dataset,
    batch_sampler=sampler,
    collate_fn=lambda b: collate_fn(b, pad_id=PAD_ID),
)

print(f"Batches per epoch: {len(sampler):,}")

# ── Quick sanity check ─────────────────────────────────────────────────────────
tokens, mask, labels = next(iter(loader))
print("tokens:", tokens.shape)   # (B, T)
print("mask:  ", mask.shape)     # (B, T)
print("labels:", labels)         # (B,)  — mix of 0s and 1s
print("true count:", (labels == 0).sum().item(),
      "| corrupt count:", (labels == 1).sum().item())


Tokenizing true strings...
Tokenizing corrupted strings...
Dataset size: 2,000,000 samples
Batches per epoch: 62,376
tokens: torch.Size([32, 50])
mask:   torch.Size([32, 50])
labels: tensor([0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0,
        0, 0, 1, 0, 1, 0, 0, 1])
true count: 16 | corrupt count: 16


In [47]:
import tqdm
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

EPOCHS = 100

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.BCEWithLogitsLoss()
model.to(DEVICE)

def train_loop(dataloader, model, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    for batch in tqdm.tqdm(dataloader, desc="Training"):
        tokens, mask, labels = batch
        tokens, mask, labels = tokens.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(tokens, mask)
        # print("Model outputs shape:", outputs.shape)  # Debug print
        # print("Model outputs:", outputs.squeeze().shape)  # Debug print
        loss = loss_fn(outputs.squeeze().float(), labels.float())
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(f"Average training loss: {avg_loss:.4f}")

for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    train_loop(loader, model, loss_fn, optimizer)


Epoch 1/100


Training:  14%|█▎        | 8533/62376 [04:57<31:16, 28.69it/s] 


KeyboardInterrupt: 